# 02 - Reinforcement Learning - Evolving MLP [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobertTLange/evosax/blob/main/examples/02_rl.ipynb)

## Installation

You will need Python 3.10 or later, and a working JAX installation. For example, you can install JAX on NVIDIA GPU with:

In [ ]:
%pip install -U "jax[cuda]"

Then, install `evosax` from PyPi:

In [ ]:
%pip install -U "evosax[examples]"

## Import

In [10]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax

In [11]:
seed = 0
key = jax.random.key(seed)

## Brax

### Ant environment

In [12]:
from evosax.problems import BraxProblem as Problem
from evosax.problems.networks import MLP, tanh_output_fn

policy = MLP(
    layer_sizes=(32, 32, 32, 32, 8),
    output_fn=tanh_output_fn,
)

problem = Problem(
    env_name="ant",
    policy=policy,
    episode_length=1000,
    num_rollouts=1,
    # use_normalize_obs=True,
)

key, subkey = jax.random.split(key)
problem_state = problem.init(key)

key, subkey = jax.random.split(key)
solution = problem.sample(subkey)

In [13]:
print(f"Number of pararmeters: {sum(leaf.size for leaf in jax.tree.leaves(solution))}")

Number of pararmeters: 4328


### Open_ES

In [14]:
def custom_metrics_fn(key, population, fitness, state, params):
    best_idx_in_generation = jnp.argmin(fitness)
    return {
        "generation_counter": state.generation_counter,
        "best_fitness_in_generation": fitness[best_idx_in_generation],
        "mean_fitness_in_generation": fitness.mean(),
        "best_fitness": state.best_fitness,
    }

In [15]:
from evosax.algorithms import Open_ES as ES

num_generations = 1000
population_size = 256

lr_schedule = optax.exponential_decay(
    init_value=0.01,
    transition_steps=num_generations,
    decay_rate=0.1,
)
std_schedule = optax.exponential_decay(
    init_value=0.05,
    transition_steps=num_generations,
    decay_rate=0.2,
)
es = ES(
    population_size=population_size,
    solution=solution,
    optimizer=optax.adam(learning_rate=lr_schedule),
    std_schedule=std_schedule,
    metrics_fn=custom_metrics_fn
)

params = es.default_params

In [19]:
import sys
sys.path.append('/home/ronedr/evolution-strategy-baselines-comparison')

from experiment.experiment import Experiment

experiment = Experiment(
    problem=problem,
    algorithm=es,
    results_dir_path="../../results",
    log_period=1,
    eval_batch_size=None,
    seed=seed
)
experiment.run(num_generations=1000)

Generation 001 | Best fitness (Train): 806.55 | Best fitness in generation (Train): 806.55 | Mean fitness (Train): 453.48 | Mean fitness (Test): -775.34
Generation 002 | Best fitness (Train): 868.44 | Best fitness in generation (Train): 868.44 | Mean fitness (Train): 622.41 | Mean fitness (Test): -846.52
Generation 003 | Best fitness (Train): 948.66 | Best fitness in generation (Train): 948.66 | Mean fitness (Train): 657.86 | Mean fitness (Test): -874.84
Generation 004 | Best fitness (Train): 948.66 | Best fitness in generation (Train): 943.06 | Mean fitness (Train): 690.27 | Mean fitness (Test): -867.05
Generation 005 | Best fitness (Train): 948.66 | Best fitness in generation (Train): 887.02 | Mean fitness (Train): 721.57 | Mean fitness (Test): -923.21
Generation 006 | Best fitness (Train): 959.78 | Best fitness in generation (Train): 959.78 | Mean fitness (Train): 738.20 | Mean fitness (Test): -900.53
Generation 007 | Best fitness (Train): 959.78 | Best fitness in generation (Train)

{'best_fitness': Array([ 806.554  ,  868.43884,  948.6633 ,  948.6633 ,  948.6633 ,
         959.7766 ,  959.7766 ,  959.7766 ,  959.7766 ,  980.6796 ,
         980.6796 ,  991.6839 ,  991.6839 , 1007.28046, 1007.28046,
        1008.5607 , 1008.5607 , 1008.5607 , 1008.5607 , 1017.993  ,
        1017.993  , 1051.5164 , 1051.5164 , 1051.5164 , 1089.0042 ,
        1089.0042 , 1089.0042 , 1089.0042 , 1089.0042 , 1182.237  ,
        1182.237  , 1182.237  , 1182.237  , 1182.237  , 1182.237  ,
        1182.237  , 1182.237  , 1182.237  , 1182.237  , 1182.237  ,
        1182.237  , 1182.237  , 1182.237  , 1182.237  , 1182.237  ,
        1263.1339 , 1263.1339 , 1263.1339 , 1413.4373 , 1413.4373 ,
        1413.4373 , 1413.4373 , 1413.4373 , 1413.4373 , 1522.5099 ,
        1522.5099 , 1522.5099 , 1522.5099 , 1522.5099 , 1522.5099 ,
        1522.5099 , 1522.5099 , 1548.5391 , 1548.5391 , 1612.4403 ,
        1612.4403 , 1612.4403 , 1612.4403 , 1612.4403 , 1612.4403 ,
        1612.4403 , 1612.4403 , 

### Run

In [16]:
def step(carry, key):
    state, params, problem_state = carry
    key_ask, key_eval, key_tell = jax.random.split(key, 3)

    population, state = es.ask(key_ask, state, params)

    fitness, problem_state, _ = problem.eval(key_eval, population, problem_state)

    state, metrics = es.tell(
        key_tell, population, -fitness, state, params
    )  # Minimize fitness

    return (state, params, problem_state), metrics

In [9]:
key, subkey = jax.random.split(key)
state = es.init(subkey, solution, params)

fitness_log = []
log_period = 1
for i in range(num_generations // log_period):
    # Train
    key, subkey = jax.random.split(key)
    keys = jax.random.split(subkey, log_period)
    (state, params, problem_state), metrics = jax.lax.scan(
        step,
        (state, params, problem_state),
        keys,
    )

    # Eval
    mean = es.get_mean(state)
    key, subkey = jax.random.split(key)
    fitness, problem_state, info = problem.eval(
        key, jax.tree.map(lambda x: x[None], mean), problem_state
    )
    print(f"Generation {(i + 1) * log_period:3d} | Mean fitness: {fitness.mean():.2f}")

Generation   1 | Mean fitness: 757.72
Generation   2 | Mean fitness: 751.98
Generation   3 | Mean fitness: 848.67
Generation   4 | Mean fitness: 932.97
Generation   5 | Mean fitness: 912.19
Generation   6 | Mean fitness: 930.23
Generation   7 | Mean fitness: 960.08
Generation   8 | Mean fitness: 954.39
Generation   9 | Mean fitness: 966.84
Generation  10 | Mean fitness: 953.69
Generation  11 | Mean fitness: 922.68
Generation  12 | Mean fitness: 971.05
Generation  13 | Mean fitness: 947.32
Generation  14 | Mean fitness: 949.50
Generation  15 | Mean fitness: 934.94
Generation  16 | Mean fitness: 947.88
Generation  17 | Mean fitness: 971.61
Generation  18 | Mean fitness: 951.97
Generation  19 | Mean fitness: 975.71
Generation  20 | Mean fitness: 957.37
Generation  21 | Mean fitness: 979.54
Generation  22 | Mean fitness: 990.20
Generation  23 | Mean fitness: 1003.03
Generation  24 | Mean fitness: 981.71
Generation  25 | Mean fitness: 997.41
Generation  26 | Mean fitness: 1001.29
Generation

## Visualize policy

In [ ]:
key, subkey = jax.random.split(key)


In [ ]:
mean = es.get_mean(state)
# mean = es._unravel_solution(state.best_solution)
fitness, _, _ = problem.eval(
    key, jax.tree.map(lambda x: x[None], mean), problem_state
)
fitness

In [9]:
from brax.io import html
from IPython.display import HTML

rollout = [
    jax.tree_util.tree_map(lambda x: x[0, 0, t], info["env_states"].pipeline_state)
    for t in range(200)
]

html_content = html.render(
    problem.env.sys.tree_replace({"opt.timestep": problem.env.dt}), rollout
)
HTML(html_content)

In [10]:
# Write to file
with open("ant_visualization.html", "w") as f:
    f.write(html_content)

print("Visualization saved to 'ant_visualization.html'")

Visualization saved to 'ant_visualization.html'
